Undersampling + Sliding Window

In [1]:
import pandas as pd
train_df = pd.read_csv(r"train.csv")
valid_df = pd.read_csv(r"val.csv")
test_df = pd.read_csv(r"test.csv")

train_df.sort_values(by='timestamp', inplace=True)
valid_df.sort_values(by='timestamp', inplace=True)
test_df.sort_values(by='timestamp', inplace=True)

cols_to_drop = ['timestamp', 'network_ips_dst', 'network_ips_src', 'network_ports_dst', 'network_ports_src']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
import torch
from torch.utils.data import Dataset
import numpy as np

class UndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, step=5):
        # Giữ lại giá trị ở định dạng numpy float32 để tiết kiệm RAM.
        # Không cần thiết tạo torch.tensor dư thừa ở đây.
        self.X = df.drop(columns=['label']).values.astype(np.float32)
        self.y = df['label'].values.astype(np.int64)
        self.time_steps = time_steps
        self.step = step
        
        print(f"Đang tính toán các cửa sổ hợp lệ (global step={step}) và Undersampling...")
        
        # mảng index step = 1
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1]
        
        # mảng index step = 5
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1]
        
        self.valid_indices = []
        
        # lặp qua từng class
        classes = np.unique(window_labels_step1)
        
        for c in classes:
            # các class hiếm dùng step =1
            if c in [9,10]:
                c_indices = all_indices_step1[window_labels_step1 == c]
            
            # các class còn lại dùng step = global step
            else:
                c_indices = all_indices_stepped[window_labels_stepped == c]
            
            # đếm số lượng cửa sổ của class đó
            count = len(c_indices)
            
            # nếu class đó có nhiều mẫu hơn ngưỡng max_samples_per_class
            if count > max_samples_per_class:
                # chọn ngẫu nhiên không hoàn lại
                c_indices = np.random.choice(c_indices, max_samples_per_class, replace=False)
                print(f"Class {c}: Giảm từ {count} xuống {max_samples_per_class} cửa sổ.")
            else:
                # nếu ít hơn thì giữ nguyên
                if c in [9,10]:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (sử dụng step=1 để bảo toàn).")
                else:
                    print(f"Class {c}: Giữ nguyên {count} cửa sổ (step={self.step}).")
                
            self.valid_indices.extend(c_indices)  # index các cửa sổ hợp lệ sau khi lọc và Undersampling
            
        # trộn đều danh sách index
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices) # Chuyển sang ndarray
        print(f"Tổng số cửa sổ sau khi lọc và Undersampling: {len(self.valid_indices)}")

    # trả về số lượng cửa sổ hợp lệ
    def __len__(self):
        return len(self.valid_indices)

    # trả về feature và label của cửa sổ trượt tại index đã lọc và xáo trộn
    def __getitem__(self, idx):
        # lấy các index đã được lọc và xáo trộn
        actual_idx = self.valid_indices[idx]
        
        # lấy feature và label của cửa sổ trượt (trích xuất dạng numpy)
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        
        # Chỉ chuyển sang tensor ngay lúc cần đưa vào thiết bị
        return torch.tensor(window_X, dtype=torch.float32), torch.tensor(label_y, dtype=torch.long)

In [3]:
class DynamicUndersampledSlidingWindowDataset(Dataset):
    def __init__(self, df, time_steps, max_samples_per_class=20000, 
                 step=5, resample_each_epoch=False, rare_classes=None):
        

        if rare_classes is None:
            rare_classes = [9, 10]
            
        feat_cols = [col for col in df.columns if col != 'label']
        self.X = torch.as_tensor(df[feat_cols].to_numpy(dtype=np.float32))
        self.y = torch.as_tensor(df['label'].to_numpy(dtype=np.int64))
        
        self.time_steps = time_steps
        self.step = step
        self.max_samples_per_class = max_samples_per_class
        self.resample_each_epoch = resample_each_epoch
        
        # Tạo 2 mảng index: 1 cái step=1 (bảo toàn), 1 cái có step (băm mỏng)
        all_indices_step1 = np.arange(0, len(self.X) - self.time_steps + 1, 1)
        window_labels_step1 = self.y[all_indices_step1 + self.time_steps - 1].numpy()
        
        all_indices_stepped = np.arange(0, len(self.X) - self.time_steps + 1, self.step)
        window_labels_stepped = self.y[all_indices_stepped + self.time_steps - 1].numpy()
        
        self.class_indices = {}
        for c in np.unique(window_labels_step1):
            if c in rare_classes:
                # Bảo toàn 100% dữ liệu cho class hiếm
                self.class_indices[c] = all_indices_step1[window_labels_step1 == c]
            else:
                # Băm mỏng theo step đối với các class đa số
                self.class_indices[c] = all_indices_stepped[window_labels_stepped == c]
            
            print(f"Class {c}: Có sẵn {len(self.class_indices[c])} cửa sổ trong Pool")
            
        self._resample()
    
    def _resample(self):
        self.valid_indices = []
        for c, c_indices in self.class_indices.items():
            count = len(c_indices)
            
            # NẾU LÀ TẬP TRAIN (có bật resample_each_epoch)
            if self.resample_each_epoch:
                if count > self.max_samples_per_class:
                    # Dư thì băm đi (Undersampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=False)
                else:
                    # Thiếu thì nhân bản (Oversampling)
                    sampled = np.random.choice(c_indices, self.max_samples_per_class, replace=True)
                    
            # NẾU LÀ TẬP VAL/TEST (Giữ nguyên gốc 100%, không thêm không bớt)
            else:
                sampled = c_indices
                
            self.valid_indices.extend(sampled)
            
        np.random.shuffle(self.valid_indices)
        self.valid_indices = np.array(self.valid_indices)
    def on_epoch_start(self):
        if self.resample_each_epoch:
            self._resample()

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        window_X = self.X[actual_idx : actual_idx + self.time_steps]
        label_y = self.y[actual_idx + self.time_steps - 1]
        return window_X, label_y

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np

MAX_SAMPLES = 20000 
TIME_STEPS = 10
BATCH_SIZE = 256
TEST_STEP_SIZE = 1
TRAIN_STEP_SIZE = 1 
NUM_CLASSES = len(train_df['label'].unique())


print(f"Khởi tạo tập Train (có Undersampling, set Train Step = {TRAIN_STEP_SIZE})...")
train_dataset = DynamicUndersampledSlidingWindowDataset(train_df, TIME_STEPS, max_samples_per_class=MAX_SAMPLES, step=TRAIN_STEP_SIZE)

print(f"Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = {TEST_STEP_SIZE})...")
# để max_samples_per_class cho tập val và test lớn để không xóa cửa sổ trượt nào
val_dataset = DynamicUndersampledSlidingWindowDataset(valid_df, TIME_STEPS, max_samples_per_class=10000000, step=TEST_STEP_SIZE)
test_dataset = DynamicUndersampledSlidingWindowDataset(test_df, TIME_STEPS, max_samples_per_class=10000000, step=TEST_STEP_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Khởi tạo tập Train (có Undersampling, set Train Step = 1)...
Class 0: Có sẵn 95760 cửa sổ trong Pool
Class 1: Có sẵn 1307 cửa sổ trong Pool
Class 2: Có sẵn 12639 cửa sổ trong Pool
Class 3: Có sẵn 12894 cửa sổ trong Pool
Class 4: Có sẵn 5278 cửa sổ trong Pool
Class 5: Có sẵn 5643 cửa sổ trong Pool
Class 6: Có sẵn 23544 cửa sổ trong Pool
Class 7: Có sẵn 1957 cửa sổ trong Pool
Khởi tạo tập Val và Test (Không Undersampling, giữ nguyên gốc, set Test Step = 1)...
Class 0: Có sẵn 20520 cửa sổ trong Pool
Class 1: Có sẵn 280 cửa sổ trong Pool
Class 2: Có sẵn 2708 cửa sổ trong Pool
Class 3: Có sẵn 2763 cửa sổ trong Pool
Class 4: Có sẵn 1131 cửa sổ trong Pool
Class 5: Có sẵn 1209 cửa sổ trong Pool
Class 6: Có sẵn 5038 cửa sổ trong Pool
Class 7: Có sẵn 419 cửa sổ trong Pool
Class 0: Có sẵn 20520 cửa sổ trong Pool
Class 1: Có sẵn 281 cửa sổ trong Pool
Class 2: Có sẵn 2709 cửa sổ trong Pool
Class 3: Có sẵn 2763 cửa sổ trong Pool
Class 4: Có sẵn 1132 cửa sổ trong Pool
Class 5: Có sẵn 1210 cửa sổ tron

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# hàm tính focal loss (hiện đang không dùng Focal Loss)
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


In [6]:
import torch
import torch.nn as nn

NUM_FEATURES = train_dataset.X.shape[1]

# tính attention cho output của BiLSTM và tạo vector ngữ cảnh (context vector)
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_outputs):
        scores = self.attention(lstm_outputs)
        weights = torch.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_outputs, dim=1)
        return context_vector, weights

# SE Block: từ đầu ra là các channel từ CNN, kênh nào chiếm tín hiệu rõ ràng thì trọng số là 1, kênh nào bị nhiễu or sai lệch thì trọng số = 0
class SEBlock1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super(SEBlock1D, self).__init__()
        # từ các channel của CNN, tính average pooling theo chiều thời gian
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # tính điểm dựa trên 1 mạng neural channel => channel/8 => channel và đi qua hàm sigmoid để tạo thang điểm 0-1 cho từng kênh
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    # b,c: batch size, số kênh
    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c) # tính toán điểm trung bình của các channel theo chiều thời gian
        y = self.fc(y).view(b, c, 1)    # tính ra thang điểm 0-1 cho từng kênh
        return x * y.expand_as(x)       # nhân điểm số này với các channel để khuếch đại kênh chuẩn và bóp nghẹt kênh bị nhiễm Drift

# residual block: CNN tích hợp SE Block và shortcut connection 
class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3):
        super(ResidualBlock1D, self).__init__()
        padding = kernel_size // 2
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding) # đi qua lớp convoluation đầu tiên
        self.gn1 = nn.GroupNorm(num_groups=8, num_channels=out_channels) # group norm: chuẩn hóa độc lập theo từng nhóm kênh
        self.relu = nn.ReLU() # hàm kích hoạt ReLU
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding) # đi qua lớp convolution thứ hai
        self.gn2 = nn.GroupNorm(num_groups=8, num_channels=out_channels) # chuẩn hóa sau lớp convolution thứ hai
        self.dropout = nn.Dropout1d(0.2)
        
        # SE Block: chấm điểm đầu ra cho các channel của CNN
        self.se = SEBlock1D(out_channels)
        
        # shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels: # nếu số kênh đầu vào khác số kênh đầu ra thì shortcut phải dùng convolution để điều chỉnh số kênh
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1),
                nn.GroupNorm(num_groups=8, num_channels=out_channels)
            )
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.gn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)
        out = self.gn2(out)
        out = self.se(out)
        out += residual  
        out = self.relu(out)
        return out


# model CNN-BiLSTM hoàn chỉnh
class CNN_BiLSTM_Attention(nn.Module):
    def __init__(self, num_features, num_classes, time_steps, hidden_size=128):
        super(CNN_BiLSTM_Attention, self).__init__()
        
        # 2 khối Residual Block
        self.res1 = ResidualBlock1D(num_features, 64)
        self.res2 = ResidualBlock1D(64, 128)
        
        # giảm chiều dài chuỗi đi một nửa
        self.pool = nn.MaxPool1d(kernel_size=2) 
        
        # cho BiLSTM nhận đầu vào là 128 channel, tạo ra output có chiều dài hiden_size*2
        self.bilstm = nn.LSTM(input_size=128, hidden_size=hidden_size, 
                              batch_first=True, bidirectional=True)
        
        # chuẩn hóa layer norm theo từng time-step
        self.layer_norm = nn.LayerNorm(hidden_size * 2)

        # tính attention score cho output của BiLSTM và tạo vector ngữ cảnh (context vector)
        self.attention = Attention(hidden_size * 2)
        self.dropout = nn.Dropout(0.5)
        
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        # Tăng cường ổn định ở bộ phân loại cuối
        self.fc_ln = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, x):
        # x ban đầu: (Batch, SeqLen, Features)
        x = x.permute(0, 2, 1) # chuyển thành (Batch, Features, SeqLen) để phù hợp với đầu vào của CNN
        
        # đi qua các khối Residual CNN + SE Block
        x = self.res1(x)
        x = self.res2(x)
        
        x = self.pool(x)
        
        # chuyển lại thành (batch, seq_len, features) để phù hợp với đầu vào của BiLSTM
        x = x.permute(0, 2, 1)
        
        out, _ = self.bilstm(x)
        
        # chuẩn hóa layer norm theo từng time-step + tính attention score để tạo vector ngữ cảnh
        out = self.layer_norm(out)
        context_vector, attn_weights = self.attention(out)
        
        # đi qua 2 lớp fully connected với dropout và layer norm để tăng cường ổn định
        out = self.dropout(context_vector)
        out = self.fc1(out) 
        out = self.fc_ln(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        return out

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {DEVICE}")
# khởi tạo mô hình
model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
print(model)

Đang sử dụng thiết bị: cuda
CNN_BiLSTM_Attention(
  (res1): ResidualBlock1D(
    (conv1): Conv1d(133, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 64, eps=1e-05, affine=True)
    (relu): ReLU()
    (conv2): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (dropout): Dropout1d(p=0.2, inplace=False)
    (se): SEBlock1D(
      (avg_pool): AdaptiveAvgPool1d(output_size=1)
      (fc): Sequential(
        (0): Linear(in_features=64, out_features=8, bias=False)
        (1): ReLU(inplace=True)
        (2): Linear(in_features=8, out_features=64, bias=False)
        (3): Sigmoid()
      )
    )
    (shortcut): Sequential(
      (0): Conv1d(133, 64, kernel_size=(1,), stride=(1,))
      (1): GroupNorm(8, 64, eps=1e-05, affine=True)
    )
  )
  (res2): ResidualBlock1D(
    (conv1): Conv1d(64, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (gn1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (relu):

In [7]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight


actual_labels = []
for idx in train_dataset.valid_indices:
    label = train_dataset.y[idx + TIME_STEPS - 1].item()
    actual_labels.append(label)

actual_labels = np.array(actual_labels)

classes_in_train = np.unique(actual_labels)
new_weights = compute_class_weight(
    class_weight='balanced', 
    classes=classes_in_train, 
    y=actual_labels
)

# căn bậc 2 trọng số
smoothed_new_weights = np.sqrt(new_weights)

# khởi tạo focal loss với trọng số đã được căn bậc 2
alpha_tensor = torch.tensor(smoothed_new_weights, dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(alpha=alpha_tensor, gamma=3.0)

print(f"Trọng số mới sau khi Undersampling và Smooth:\n{alpha_tensor.cpu().numpy()}")

Trọng số mới sau khi Undersampling và Smooth:
[0.45560822 3.8998313  1.2540858  1.241623   1.9406576  1.8768457
 0.9188476  3.187045  ]


In [8]:
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score
# tạo folder để lưu model
import os
model_dir = "model_final"
os.makedirs(model_dir, exist_ok=True)

model = CNN_BiLSTM_Attention(num_features=NUM_FEATURES, num_classes=NUM_CLASSES, time_steps=TIME_STEPS).to(DEVICE)
# dùng Cross Entropy với Label Smoothing = 0.1
# label smoothing: thay vì one-hot encoding với 1 cho nhãn đúng và 0 cho nhãn sai thì ta giảm đi xuống mức 0.933 cho nhãn đúng để chống over-confidence
weights_tensor = torch.tensor(smoothed_new_weights, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# dùng adam. Weight decay: hiệu chỉnh l2, giúp giảm giá trị trọng số đi
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-3)

# giảm learning rate đi một nửa nếu val macro f1 không cải thiện sau 2 epochs (lưu ý đổi mode thành 'max')
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 60
best_val_f1 = 0.0
patience_counter = 0
EARLY_STOP_PATIENCE = 20

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_train = 0
    
    all_train_preds = []
    all_train_targets = []
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False)
    for inputs, labels in loop:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        
        # gradient clipping: gom tất cả cá gradient của mô hình lại thành 1 véc tơ duy nhất => tính l2 norm => nếu độ lớn vượt quá max_norm thì scale tất cả gradient xuống để l2 norm bằng max_norm
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        
        all_train_preds.extend(predicted.cpu().numpy())
        all_train_targets.extend(labels.cpu().numpy())
        
        loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / total_train
    train_macro_f1 = f1_score(all_train_targets, all_train_preds, average='macro', zero_division=0)
    
    model.eval()
    val_loss = 0.0
    total_val = 0
    
    all_val_preds = []
    all_val_targets = []
    
    # nếu là epoch chẵn thì không đánh giá trên tập val để tiết kiệm thời gian, chỉ đánh giá trên tập val ở các epoch lẻ

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Validation]", leave=False):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            
            total_val += labels.size(0)
            
            all_val_preds.extend(predicted.cpu().numpy())
            all_val_targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / total_val
    val_macro_f1 = f1_score(all_val_targets, all_val_preds, average='macro', zero_division=0)
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Train Loss: {avg_train_loss:.4f}, Train Macro F1: {train_macro_f1:.4f} | Val Loss: {avg_val_loss:.4f}, Val Macro F1: {val_macro_f1:.4f}")
    
    # Đưa val_macro_f1 vào scheduler (trước đó là avg_val_loss)
    scheduler.step(val_macro_f1)
    
    # Lưu model dựa trên Macro F1
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), "model_final/best_cnn_bilstm_exper_3.pth")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\n[Early Stopping] Model không cải thiện Macro F1 sau {EARLY_STOP_PATIENCE} epochs. Dừng huấn luyện.")
            break

print("\n--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---")
model.load_state_dict(torch.load("model_final/best_cnn_bilstm_exper_3.pth", weights_only=True))
model.eval()

all_test_preds = []
all_test_targets = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Final Test]", leave=False):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_test_preds.extend(predicted.cpu().numpy())
        all_test_targets.extend(labels.cpu().numpy())

print(classification_report(all_test_targets, all_test_preds, zero_division=0, digits=4))

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/60] - Train Loss: 0.6803, Train Macro F1: 0.8281 | Val Loss: 0.8609, Val Macro F1: 0.6821


Epoch [2/60] - Train Loss: 0.5476, Train Macro F1: 0.9629 | Val Loss: 0.8655, Val Macro F1: 0.7442


Epoch [3/60] - Train Loss: 0.5342, Train Macro F1: 0.9728 | Val Loss: 0.7874, Val Macro F1: 0.7292


Epoch [4/60] - Train Loss: 0.5302, Train Macro F1: 0.9749 | Val Loss: 0.8374, Val Macro F1: 0.7432


...

Epoch [24/60] - Train Loss: 0.5054, Train Macro F1: 0.9843 | Val Loss: 0.8403, Val Macro F1: 0.7340


Epoch [25/60] - Train Loss: 0.5056, Train Macro F1: 0.9845 | Val Loss: 0.8539, Val Macro F1: 0.7141


Epoch [26/60] - Train Loss: 0.5050, Train Macro F1: 0.9855 | Val Loss: 0.8458, Val Macro F1: 0.7471


Epoch [27/60] - Train Loss: 0.5051, Train Macro F1: 0.9856 | Val Loss: 0.8458, Val Macro F1: 0.7642

[Early Stopping] Model không cải thiện Macro F1 sau 20 epochs. Dừng huấn luyện.

--- BÁO CÁO PHÂN LOẠI TRÊN TẬP TEST ---


              precision    recall  f1-score   support

           0     0.9817    0.9985    0.9900     20520
           1     0.4432    0.9715    0.6087       281
           2     0.5101    0.4097    0.4545      2709
           3     0.5323    0.4658    0.4968      2763
           4     0.8673    0.9991    0.9286      1132
           5     0.9732    0.3000    0.4586      1210
           6     0.8636    1.0000    0.9268      5039
           7     0.8562    0.9786    0.9133       420

    accuracy                         0.8835     34074
   macro avg     0.7534    0.7654    0.7222     34074
weighted avg     0.8802    0.8835    0.8731     34074



In [8]:
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm

print("Đang tải lại model tốt nhất (best_cnn_bilstm_best.pth) để tính AUC-ROC...")

# Khởi tạo lại kiến trúc model và load trọng số
model_eval = CNN_BiLSTM_Attention(
    num_features=NUM_FEATURES, 
    num_classes=NUM_CLASSES, 
    time_steps=TIME_STEPS
).to(DEVICE)

model_eval.load_state_dict(torch.load("model_final/best_cnn_bilstm_exper_3.pth", map_location=DEVICE, weights_only=True))
model_eval.eval()

all_test_probas = []
all_test_targets_auc = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="[Tính xác suất trên tập Test]", leave=False):
        inputs = inputs.to(DEVICE)
        
        # Lấy output (logit) từ model
        outputs = model_eval(inputs)
        
        # Áp dụng hàm softmax để biến logit thành xác suất dự đoán cho từng nhãn
        probas = F.softmax(outputs, dim=1)
        
        all_test_probas.extend(probas.cpu().numpy())
        all_test_targets_auc.extend(labels.numpy())

all_test_targets_auc = np.array(all_test_targets_auc)
all_test_probas = np.array(all_test_probas)

# Tính toán Macro AUC-ROC OVR
auc_roc = roc_auc_score(all_test_targets_auc, all_test_probas, multi_class='ovr', average='macro')

print(f"\n=> AUC-ROC (Macro, OVR) của mô hình CNN-BiLSTM: {auc_roc:.4f}")

Đang tải lại model tốt nhất (best_cnn_bilstm_best.pth) để tính AUC-ROC...



=> AUC-ROC (Macro, OVR) của mô hình CNN-BiLSTM: 0.9415
